In [22]:
import pandas as pd
import numpy as np
import torch
import re
from transformers import AutoTokenizer, AutoModel
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [9]:
tokenizer = AutoTokenizer.from_pretrained("cointegrated/rubert-tiny2")
model = AutoModel.from_pretrained("cointegrated/rubert-tiny2")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 4761.43it/s]
[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
def get_embeddings(texts, batch_size=64):
    model.eval()
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size].astype(str).tolist()
        encoded = tokenizer(batch_texts, padding=True, truncation=True, max_length=512, return_tensors='pt')
        encoded = {k: v.to(device) for k, v in encoded.items()}
        with torch.no_grad():
            model_output = model(**encoded)
            # Берем вектор CLS-токена, отвечающий за смысл всей фразы
            embeddings = model_output.last_hidden_state[:, 0, :]
            embeddings = torch.nn.functional.normalize(embeddings)
            all_embeddings.append(embeddings.cpu().numpy())
    return np.vstack(all_embeddings)

In [11]:
target_dataset_df = pd.read_csv('target_dataset.csv')

text_embeddings = get_embeddings(target_dataset_df['real_description'].fillna('').values)

emb_cols = [f'emb_{i}' for i in range(text_embeddings.shape[1])]
df_embeddings = pd.DataFrame(text_embeddings, columns=emb_cols, index=target_dataset_df.index)

print("Размерность полученной таблицы эмбеддингов:", df_embeddings.shape)

Размерность полученной таблицы эмбеддингов: (2657, 312)


In [12]:
numeric_features = ['price', 'deposit', 'commission']

X = pd.concat([target_dataset_df[numeric_features], df_embeddings], axis=1)

y = target_dataset_df['target']

print(f"Формат матрицы признаков X: {X.shape}")
print(f"Формат целевой переменной y: {y.shape}")

Формат матрицы признаков X: (2657, 315)
Формат целевой переменной y: (2657,)


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Обучающая выборка: {X_train.shape[0]} примеров")
print(f"Тестовая выборка: {X_test.shape[0]} примеров")

Обучающая выборка: 2125 примеров
Тестовая выборка: 532 примеров


In [15]:
model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.08,
    depth=6,
    auto_class_weights='Balanced',  # Включаем балансировку под редкий класс LLM
    random_seed=42,
    verbose=50  # Печатаем лог каждые 50 итераций
)

In [16]:
model.fit(X_train, y_train)

0:	learn: 1.0442338	total: 127ms	remaining: 38.1s
50:	learn: 0.4040235	total: 1.83s	remaining: 8.96s
100:	learn: 0.2575641	total: 3.48s	remaining: 6.86s
150:	learn: 0.1754506	total: 4.94s	remaining: 4.87s
200:	learn: 0.1321758	total: 6.38s	remaining: 3.14s
250:	learn: 0.1072549	total: 7.85s	remaining: 1.53s
299:	learn: 0.0893138	total: 9.26s	remaining: 0us


CatBoostClassifier(auto_class_weights='Balanced', depth=6, iterations=300, learning_rate=0.08, random_seed=42, verbose=50)

In [17]:
y_pred = model.predict(X_test)

print("\n--- Отчет по метрикам классификации ---")
print(classification_report(y_test, y_pred, target_names=['Собственник (0)', 'Риелтор (1)', 'LLM (2)']))


--- Отчет по метрикам классификации ---
                 precision    recall  f1-score   support

Собственник (0)       0.68      0.93      0.78       108
    Риелтор (1)       0.95      0.88      0.91       400
        LLM (2)       0.62      0.42      0.50        24

       accuracy                           0.86       532
      macro avg       0.75      0.74      0.73       532
   weighted avg       0.88      0.86      0.87       532



In [ ]:
# 1. Генерируем явные текстовые фичи
def extract_text_features(df_src):
    feats = pd.DataFrame(index=df_src.index)
    text_s = df_src['real_description'].fillna('').astype(str)
    
    feats['char_len'] = text_s.apply(len)
    feats['word_count'] = text_s.apply(lambda x: len(x.split()))
    feats['avg_word_len'] = feats['char_len'] / (feats['word_count'] + 1e-5)
    
    # Специфичные маркировки ИИ (списки, эмодзи, структуры)
    feats['exclamation_mark_count'] = text_s.apply(lambda x: x.count('!'))
    feats['bullet_count'] = text_s.apply(lambda x: len(re.findall(r'[\bullet\-\*\d+\.]', x)))
    feats['has_emoji'] = text_s.apply(lambda x: int(bool(re.search(r'[\U00010000-\U0010ffff]', x))))
    
    return feats

# Вычисляем дополнительные фичи
df_text_feats = extract_text_features(target_dataset_df)

# 2. Собираем новую расширенную матрицу X
X_enhanced = pd.concat([
    target_dataset_df[['price', 'deposit', 'commission']], 
    df_text_feats, 
    df_embeddings
], axis=1)

# 3. Переразбиваем выборку
X_train, X_test, y_train, y_test = train_test_split(
    X_enhanced, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

# 4. Задаем ручные веса классов (увеличиваем приоритет LLM)
# Порядок классов: [0 (Собственник), 1 (Риелтор), 2 (LLM)]
class_weights = [1.0, 1.0, 4.0] 

model_v2 = CatBoostClassifier(
    iterations=400,
    learning_rate=0.06,
    depth=6,
    class_weights=class_weights,  # Применяем усиленный вес для LLM
    random_seed=42,
    verbose=0
)

model_v2.fit(X_train, y_train)

# 5. Проверяем новые результаты
y_pred_v2 = model_v2.predict(X_test)
print("--- Обновленный отчет с фичами текста и кастомными весами ---")
print(classification_report(y_test, y_pred_v2, target_names=['Собственник (0)', 'Риелтор (1)', 'LLM (2)']))

--- Обновленный отчет с фичами текста и кастомными весами ---
                 precision    recall  f1-score   support

Собственник (0)       0.80      0.78      0.79       136
    Риелтор (1)       0.92      0.93      0.92       499
        LLM (2)       0.65      0.57      0.61        30

       accuracy                           0.88       665
      macro avg       0.79      0.76      0.77       665
   weighted avg       0.88      0.88      0.88       665



In [24]:
import pandas as pd

# Получаем важность признаков
feature_importances = pd.Series(
    model_v2.get_feature_importance(), 
    index=X_enhanced.columns
).sort_values(ascending=False)

print("--- ТОП-10 самых важных признаков ---")
print(feature_importances.head(10))

--- ТОП-10 самых важных признаков ---
commission      26.473796
has_emoji        7.122821
word_count       2.625796
bullet_count     1.862796
emb_278          1.341472
emb_101          1.119055
emb_134          1.005057
char_len         0.877282
emb_17           0.821498
emb_261          0.786189
dtype: float64


In [ ]:
# 1. Расширенная генерация фич
def extract_advanced_text_features(df_src):
    feats = pd.DataFrame(index=df_src.index)
    text_s = df_src['real_description'].fillna('').astype(str)
    
    # Длина и слова
    feats['char_len'] = text_s.apply(len)
    feats['word_count'] = text_s.apply(lambda x: len(x.split()))
    feats['avg_word_len'] = feats['char_len'] / (feats['word_count'] + 1e-5)
    
    # Форматирование и пунктуация
    feats['exclamation_mark_count'] = text_s.apply(lambda x: x.count('!'))
    feats['bullet_count'] = text_s.apply(lambda x: len(re.findall(r'[\bullet\-\*\d+\.]', x)))
    feats['has_emoji'] = text_s.apply(lambda x: int(bool(re.search(r'[\U00010000-\U0010ffff]', x))))
    
    # Новые фичи-киллеры для отлова ИИ и капса риелторов
    feats['caps_ratio'] = text_s.apply(lambda x: sum(1 for c in x if c.isupper()) / (len(x) + 1e-5))
    feats['paragraph_count'] = text_s.apply(lambda x: x.count('\n') + 1)
    feats['digits_count'] = text_s.apply(lambda x: sum(1 for c in x if c.isdigit()))
    
    return feats

# Вычисляем признаки
df_adv_feats = extract_advanced_text_features(target_dataset_df)

# Собираем общую матрицу X
X_v3 = pd.concat([
    target_dataset_df[['price', 'deposit', 'commission']], 
    df_adv_feats, 
    df_embeddings
], axis=1)

# 2. Разделение выборки
X_train, X_test, y_train, y_test = train_test_split(
    X_v3, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

# 3. Обучаем CatBoost
model_v3 = CatBoostClassifier(
    iterations=700,
    learning_rate=0.05,
    depth=6,
    class_weights=[1.0, 1.0, 4.5],
    random_seed=42,
    verbose=50
)
model_v3.fit(X_train, y_train)

# 4. Настройка порога вероятностей для класса LLM (класс 2)
probabilities = model_v3.predict_proba(X_test)

print("--- Поиск оптимального порога для ИИ (класс 2) ---")
best_threshold = 0.33
best_llm_f1 = 0

for threshold in np.arange(0.15, 0.50, 0.05):
    custom_preds = []
    for proba in probabilities:
        p_owner, p_realtor, p_llm = proba
        # Если вероятность ИИ выше порога -> ставим класс 2
        if p_llm >= threshold:
            custom_preds.append(2)
        else:
            custom_preds.append(0 if p_owner >= p_realtor else 1)
            
    current_f1 = f1_score(y_test, custom_preds, average=None)[2]
    print(f"Порог P(LLM) >= {threshold:.2f} | F1-Score (LLM): {current_f1:.4f}")
    
    if current_f1 > best_llm_f1:
        best_llm_f1 = current_f1
        best_threshold = threshold

print(f"\nСамый лучший порог для ИИ: {best_threshold:.2f}")

# 5. Итоговый отчет с найденным лучшим порогом
final_preds = []
for proba in probabilities:
    p_owner, p_realtor, p_llm = proba
    if p_llm >= best_threshold:
        final_preds.append(2)
    else:
        final_preds.append(0 if p_owner >= p_realtor else 1)

print("\n--- Финальный отчет с кастомным порогом ---")
print(classification_report(y_test, final_preds, target_names=['Собственник (0)', 'Риелтор (1)', 'LLM (2)']))

0:	learn: 1.0511132	total: 102ms	remaining: 1m 11s
50:	learn: 0.4258186	total: 2.38s	remaining: 30.3s
100:	learn: 0.3035592	total: 4.44s	remaining: 26.4s
150:	learn: 0.2347711	total: 5.96s	remaining: 21.7s
200:	learn: 0.1855403	total: 7.4s	remaining: 18.4s
250:	learn: 0.1521246	total: 8.87s	remaining: 15.9s
300:	learn: 0.1292589	total: 10.3s	remaining: 13.7s
350:	learn: 0.1138510	total: 11.8s	remaining: 11.7s
400:	learn: 0.1003479	total: 13.7s	remaining: 10.2s
450:	learn: 0.0891499	total: 15.6s	remaining: 8.62s
500:	learn: 0.0801385	total: 17.3s	remaining: 6.85s
550:	learn: 0.0722057	total: 18.8s	remaining: 5.08s
600:	learn: 0.0659434	total: 20.3s	remaining: 3.34s
650:	learn: 0.0601184	total: 21.8s	remaining: 1.64s
699:	learn: 0.0554472	total: 23.4s	remaining: 0us
--- Поиск оптимального порога для ИИ (класс 2) ---
Порог P(LLM) >= 0.15 | F1-Score (LLM): 0.5455
Порог P(LLM) >= 0.20 | F1-Score (LLM): 0.6545
Порог P(LLM) >= 0.25 | F1-Score (LLM): 0.6792
Порог P(LLM) >= 0.30 | F1-Score (LLM